In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS capgeminipro.retail_silver;

In [0]:
%sql
USE capgeminipro.retail_silver;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_customers AS

SELECT DISTINCT

    CAST(CustomerID AS INT) AS CustomerID,

    INITCAP(TRIM(CustomerName)) AS CustomerName,

    LOWER(TRIM(Email)) AS Email,

    TRIM(City) AS City,

    TRIM(Address) AS Address,

    TO_DATE(LastUpdated, 'dd-MM-yyyy') AS LastUpdated

FROM capgeminipro.retail_bronze.bronze_customers;

In [0]:
%sql

CREATE OR REPLACE TABLE silver_customers AS

SELECT DISTINCT

    CAST(CustomerID AS INT) AS CustomerID,

    INITCAP(TRIM(CustomerName)) AS CustomerName,

    LOWER(TRIM(Email)) AS Email,

    TRIM(City) AS City,

    TRIM(Address) AS Address,

    TO_DATE(LastUpdated, 'dd-MM-yyyy') AS LastUpdated

FROM capgeminipro.retail_bronze.bronze_customers1;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_products AS

SELECT DISTINCT

    CAST(ProductID AS INT) AS ProductID,

    TRIM(ProductName) AS ProductName,

    TRIM(Category) AS Category,

    CAST(UnitPrice AS DECIMAL(10,2)) AS UnitPrice

FROM capgeminipro.retail_bronze.bronze_products;

In [0]:
%sql
SELECT * FROM rejected_products;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_products_clean AS

SELECT *

FROM silver_products

WHERE UnitPrice > 0;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_stores AS

SELECT DISTINCT

    CAST(StoreID AS INT) AS StoreID,

    INITCAP(TRIM(StoreName)) AS StoreName,

    TRIM(Region) AS Region

FROM capgeminipro.retail_bronze.bronze_stores;

In [0]:
%sql
CREATE OR REPLACE TABLE rejected_stores AS

SELECT *

FROM silver_stores

WHERE Region IS NULL;

In [0]:
%sql
SELECT * FROM rejected_stores;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_stores_clean AS

SELECT *

FROM silver_stores

WHERE Region IS NOT NULL;

In [0]:
%sql
SELECT * FROM silver_stores_clean;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_sales AS

SELECT DISTINCT

    CAST(TransactionID AS INT) AS TransactionID,

    CAST(CustomerID AS INT) AS CustomerID,

    CAST(ProductID AS INT) AS ProductID,

    CAST(StoreID AS INT) AS StoreID,

    CAST(Quantity AS INT) AS Quantity,

    TO_DATE(TxnDate, 'dd-MM-yyyy') AS TxnDate

FROM capgeminipro.retail_bronze.bronze_sales;

In [0]:
%sql
CREATE OR REPLACE TABLE rejected_sales AS

SELECT *

FROM silver_sales

WHERE Quantity <= 0;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_sales_clean AS

SELECT *

FROM silver_sales

WHERE Quantity > 0;

In [0]:
%sql
SELECT * FROM silver_sales_clean;

In [0]:
%sql
CREATE OR REPLACE TABLE rejected_sales_invalid_customer AS

SELECT s.*

FROM silver_sales_clean s

LEFT JOIN silver_customers sc
ON s.CustomerID = sc.CustomerID

WHERE sc.CustomerID IS NULL;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_sales_final AS

SELECT s.*

FROM silver_sales_clean s

INNER JOIN silver_customers sc
ON s.CustomerID = sc.CustomerID;